# RHI Live Runtime v13 — Model-Origin Recursive Agent

Δ **Purpose:** v11 proved the controller shape, but the actual LLM branch generation failed with `AttributeError`, so deterministic fallback text collapsed as Ψ. v13 fixes that.

Core correction:

$$
\text{fallback output} \neq \Psi
$$

If `LOAD_REAL_MODEL=True`, the runtime must prove the answer came from the model before collapse. Otherwise the state is $\Omega_{\text{model}}$, not fake success.

Pipeline:

$$
Q \rightarrow C_Q \rightarrow B_i \rightarrow A_i \rightarrow G_{\text{origin}} \rightarrow G_{\text{grounding}} \rightarrow \Psi \text{ or } \Omega
$$

Where:

- $Q$ = prompt
- $C_Q$ = need-slot contract
- $B_i$ = recursive KRRB branch candidates
- $A_i$ = operational audit
- $G_{\text{origin}}$ = model-origin gate
- $G_{\text{grounding}}$ = prompt/contract grounding gate


In [1]:
# Optional install cell.
# Run only if a package is missing.
#
# %pip install -U pandas numpy torch transformers accelerate safetensors sentencepiece


In [2]:
from __future__ import annotations

import os
import re
import json
import math
import time
import uuid
import random
import traceback
import hashlib
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v13_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v13_" + uuid.uuid4().hex[:10]

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("RUN_ID:", RUN_ID)


ROOT: D:\Nexus\Nexus Mark 9\NoteBooks
OUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs
RUN_ID: rhi_v13_29cb18de15


## Runtime configuration

Use a local model folder or a Hugging Face model ID.

Good RTX 4060 starter:

```python
MODEL_ID_OR_PATH = "Qwen/Qwen2.5-1.5B-Instruct"
```

If you already downloaded the model into the same folder as the notebook:

```python
MODEL_ID_OR_PATH = "./Qwen2.5-1.5B-Instruct"
```


In [3]:
MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")

LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True

MAX_NEW_TOKENS = 420
TEMPERATURE = 0.35
TOP_P = 0.90

MAX_RECURSION_DEPTH = 2

PSI_MIN = 0.58
MARGIN_MIN = 0.045
PROMPT_FIT_MIN = 0.28
QUALITY_MIN = 0.42
BOILERPLATE_MAX = 0.30
TRACE_MIN = 0.45

print("MODEL_ID_OR_PATH:", MODEL_ID_OR_PATH)
print("LOAD_REAL_MODEL:", LOAD_REAL_MODEL)
print("REQUIRE_MODEL_FOR_PSI:", REQUIRE_MODEL_FOR_PSI)


MODEL_ID_OR_PATH: Qwen/Qwen2.5-1.5B-Instruct
LOAD_REAL_MODEL: True
REQUIRE_MODEL_FOR_PSI: True


In [4]:
# Model loading and smoke test.
# v13 fixes the common AttributeError source:
# do NOT assume model.device exists. Use next(model.parameters()).device.

tokenizer = None
model = None
MODEL_READY = False
MODEL_GENERATION_READY = False
MODEL_ERROR = None
DEVICE_INFO = {}

def infer_model_device():
    import torch
    if model is None:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def try_load_model(model_id_or_path: str) -> bool:
    global tokenizer, model, MODEL_READY, MODEL_ERROR, DEVICE_INFO

    if not LOAD_REAL_MODEL:
        MODEL_ERROR = "LOAD_REAL_MODEL=False"
        print("Model loading disabled.")
        return False

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        DEVICE_INFO["torch_version"] = torch.__version__
        DEVICE_INFO["cuda_available"] = bool(torch.cuda.is_available())
        DEVICE_INFO["device_count"] = int(torch.cuda.device_count())
        if torch.cuda.is_available():
            DEVICE_INFO["gpu_name"] = torch.cuda.get_device_name(0)
            DEVICE_INFO["cuda_version"] = torch.version.cuda

        print("Torch/CUDA:", DEVICE_INFO)

        tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, trust_remote_code=True)
        if tokenizer.pad_token_id is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32

        try:
            model = AutoModelForCausalLM.from_pretrained(
                model_id_or_path,
                trust_remote_code=True,
                dtype=dtype,
                device_map="auto" if torch.cuda.is_available() else None,
                low_cpu_mem_usage=True,
            )
        except TypeError:
            # Older transformers uses torch_dtype.
            model = AutoModelForCausalLM.from_pretrained(
                model_id_or_path,
                trust_remote_code=True,
                torch_dtype=dtype,
                device_map="auto" if torch.cuda.is_available() else None,
                low_cpu_mem_usage=True,
            )

        if not torch.cuda.is_available():
            model.to(torch.device("cpu"))

        model.eval()
        MODEL_READY = True
        MODEL_ERROR = None
        print("MODEL_READY:", MODEL_READY)
        print("INFER_DEVICE:", infer_model_device())
        return True

    except Exception as e:
        MODEL_READY = False
        MODEL_ERROR = "".join(traceback.format_exception_only(type(e), e)).strip()
        print("MODEL LOAD FAILED.")
        print(MODEL_ERROR)
        return False

def raw_model_generate(messages: List[Dict[str, str]], max_new_tokens: int = 80) -> str:
    import torch
    if not MODEL_READY:
        raise RuntimeError("Model is not loaded.")

    device = infer_model_device()

    if hasattr(tokenizer, "apply_chat_template"):
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )
        input_ids = input_ids.to(device)

        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        gen = out[0][input_ids.shape[-1]:]
        return tokenizer.decode(gen, skip_special_tokens=True).strip()

    text = "\n\n".join([m["role"].upper() + ":\n" + m["content"] for m in messages]) + "\n\nASSISTANT:\n"
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

_ = try_load_model(MODEL_ID_OR_PATH)

try:
    smoke = raw_model_generate([
        {"role": "system", "content": "You are a runtime smoke test."},
        {"role": "user", "content": "Reply with one short sentence containing the word READY."},
    ], max_new_tokens=40)
    MODEL_GENERATION_READY = True
    print("MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
    print("SMOKE:", smoke)
except Exception as e:
    MODEL_GENERATION_READY = False
    MODEL_ERROR = "".join(traceback.format_exception_only(type(e), e)).strip()
    print("MODEL GENERATION FAILED.")
    print(MODEL_ERROR)
    print(traceback.format_exc())


Torch/CUDA: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'gpu_name': 'NVIDIA GeForce RTX 4060', 'cuda_version': '12.6'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL_READY: True
INFER_DEVICE: cuda:0
MODEL GENERATION FAILED.
AttributeError
Traceback (most recent call last):
  File "C:\Users\Developer\anaconda3\envs\nexus-ultimate\Lib\site-packages\transformers\tokenization_utils_base.py", line 275, in __getattr__
    return self.data[item]
           ~~~~~~~~~^^^^^^
KeyError: 'shape'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Developer\AppData\Local\Temp\ipykernel_26632\3374462837.py", line 126, in <module>
    smoke = raw_model_generate([
            ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Developer\AppData\Local\Temp\ipykernel_26632\3374462837.py", line 100, in raw_model_generate
    out = model.generate(
          ^^^^^^^^^^^^^^^
  File "C:\Users\Developer\anaconda3\envs\nexus-ultimate\Lib\site-packages\torch\utils\_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Developer\anaconda3\en

## Core language filters

This separates framework vocabulary from operational agreement.

The v4/v5 false-lock was:

$$
\text{shared Nexus words} \Rightarrow \text{fake consensus}
$$

v13 blocks that by treating Nexus surface terms as a stop-band.


In [5]:
STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","of","to","in","on","for","with","by","as",
    "is","are","was","were","be","being","been","it","this","that","these","those","from","at",
    "into","out","about","so","because","therefore","than","not","no","yes","do","does","did",
    "can","could","should","would","will","just","they","them","their","you","your","we","our",
    "i","me","my","he","she","his","her","its","what","how","why","when"
}

NEXUS_SURFACE_TERMS = {
    "nexus","contract","carrier","domain","boundary","collapse","shape","value","slot","need",
    "forbidden","neighbor","operational","recursive","recursion","krrb","omega","psi","field",
    "fold","runtime","phase","lock","audit","trace","signal","evidence","branch","repair",
    "candidate","construct","verify","counter","prompt"
}

BOILERPLATE_PHRASES = [
    "the prompt is asking",
    "contract-first answer",
    "the correct flow is prompt",
    "answer should not be a noun lookup",
    "construct the inverse shape",
    "forms a need-slot before acting",
]

def words(text: str, remove_nexus_surface: bool = False) -> List[str]:
    toks = re.findall(r"[a-zA-Z0-9_ΔΨΩ⊕↻⊥]+", str(text).lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    if remove_nexus_surface:
        toks = [t for t in toks if t not in NEXUS_SURFACE_TERMS]
    return toks

def wordset(text: str, remove_nexus_surface: bool = False) -> set:
    return set(words(text, remove_nexus_surface=remove_nexus_surface))

def jaccard_text(a: str, b: str, remove_nexus_surface: bool = False) -> float:
    wa, wb = wordset(a, remove_nexus_surface), wordset(b, remove_nexus_surface)
    if not wa and not wb:
        return 1.0
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / max(1, len(wa | wb))

def contains_any(text: str, terms: List[str]) -> bool:
    s = str(text).lower()
    return any(str(t).lower() in s for t in terms)

def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, float(x)))

def harmonic_mean(vals: List[float], eps: float = 1e-9) -> float:
    vals = [max(eps, float(v)) for v in vals]
    return len(vals) / sum(1.0 / v for v in vals)

def field_hit_score(text: str, field_terms: List[str], max_terms: int = 10) -> float:
    if not field_terms:
        return 0.5
    s = str(text).lower()
    uniq = []
    for t in field_terms:
        t = str(t).lower().strip()
        if t and t not in uniq:
            uniq.append(t)
    uniq = uniq[:max_terms]
    hits = sum(1 for term in uniq if term in s)
    return clamp(hits / max(1, len(uniq)))

def boilerplate_penalty(text: str) -> float:
    s = str(text).lower()
    phrase_hits = sum(1 for p in BOILERPLATE_PHRASES if p in s)
    repeated_contract_words = sum(s.count(t) for t in ["contract", "collapse", "slot", "operational", "branch"])
    phrase_pen = phrase_hits / max(1, len(BOILERPLATE_PHRASES))
    repeat_pen = clamp(max(0, repeated_contract_words - 8) / 20)
    return clamp(0.70 * phrase_pen + 0.30 * repeat_pen)


In [6]:
SHAPE_TEMPLATES = {
    "CONTRACT": {
        "triggers": ["contract", "before", "intent", "tool", "agent", "plan", "spec", "interface"],
        "needs": ["intent", "boundary", "tool", "before", "select", "gate"]
    },
    "GROOVE": {
        "triggers": ["train", "lora", "qlora", "adapter", "fine tune", "weights", "groove", "model"],
        "needs": ["adapter", "low-rank", "weights", "delta", "dataset", "loss", "eval"]
    },
    "SEARCH": {
        "triggers": ["search", "retrieve", "retrieval", "find", "query", "lookup", "index", "rag"],
        "needs": ["query", "retrieve", "candidate", "rank", "verify", "evidence"]
    },
    "REPAIR": {
        "triggers": ["fix", "repair", "error", "failed", "broken", "bug", "traceback", "syntaxerror", "nameerror"],
        "needs": ["failure", "cause", "patch", "test", "rerun", "trace"]
    },
    "MEMORY": {
        "triggers": ["remember", "memory", "recall", "lost", "state", "context", "continuity"],
        "needs": ["state", "trace", "retrieve", "preserve", "update", "continuity"]
    },
    "BOUNDARY": {
        "triggers": ["boundary", "limit", "forbidden", "constraint", "safety", "gate", "reject"],
        "needs": ["boundary", "reject", "constraint", "preserve", "violate", "gate"]
    },
    "RECURSE": {
        "triggers": ["recursive", "recursion", "again", "loop", "fold", "iterate", "turn"],
        "needs": ["recursive", "branch", "feedback", "repair", "collapse", "omega"]
    },
    "TOOL": {
        "triggers": ["tool", "api", "call", "function", "agent", "execute", "act"],
        "needs": ["tool", "input", "output", "side-effect", "verify"]
    },
}

def detect_shape_template(prompt: str) -> List[str]:
    p = str(prompt).lower()
    active = []
    for name, cfg in SHAPE_TEMPLATES.items():
        if any(t in p for t in cfg["triggers"]):
            active.append(name)
    return active or ["GENERAL"]

def shape_mass(text: str, active_templates: List[str]) -> Dict[str, float]:
    masses = {}
    for name in active_templates:
        if name == "GENERAL":
            continue
        needs = SHAPE_TEMPLATES[name]["needs"]
        masses[name] = sum(1 for n in needs if n.lower() in str(text).lower()) / max(1, len(needs))
    if not masses:
        masses["GENERAL"] = 0.5
    return masses

def shape_score(text: str, active_templates: List[str]) -> float:
    masses = shape_mass(text, active_templates)
    return clamp(sum(masses.values()) / max(1, len(masses)))


## Need-slot contract

The contract is the inverse cavity. It is not the answer.

$$
C_Q = (N_Q, F_Q, B_Q, T_Q, \Psi_Q)
$$

v13 uses this as the gate input, not as decorative text.


In [7]:
@dataclass
class NeedSlotContract:
    prompt: str
    active_templates: List[str]
    inverse_need: str
    preserved_function: str
    boundary_conditions: List[str]
    domain_carrier: List[str]
    forbidden_neighbors: List[str]
    collapse_target: str
    repair_history: List[Dict[str, Any]] = field(default_factory=list)

def extract_domain_terms(prompt: str, max_terms: int = 14) -> List[str]:
    ws = words(prompt, remove_nexus_surface=True)
    counts = Counter(ws)
    return [w for w, _ in counts.most_common(max_terms)]

def infer_forbidden_neighbors(active: List[str]) -> List[str]:
    forb = set()
    if "TOOL" in active or "CONTRACT" in active:
        forb.update(["tool-first action", "premature execution", "api reflex", "surface task completion"])
    if "GROOVE" in active:
        forb.update(["full retrain reflex", "weight churn", "dataset worship", "loss-only tuning"])
    if "SEARCH" in active:
        forb.update(["noun lookup", "keyword matching", "unverified retrieval", "search without verifier"])
    if "REPAIR" in active:
        forb.update(["blanket rewrite", "threshold fiddling", "silent failure", "patch without test"])
    if "MEMORY" in active:
        forb.update(["stateless answer", "context amnesia", "surface recall", "summary as memory"])
    if "BOUNDARY" in active:
        forb.update(["unsafe override", "constraint erasure", "boundary confusion"])
    if "RECURSE" in active:
        forb.update(["linear pipeline", "single branch", "dead loop", "nested sweep masquerading as recursion"])
    if not forb:
        forb.update(["surface label", "generic explanation", "noun-only answer"])
    return sorted(forb)

def build_contract(prompt: str, repair_history: Optional[List[Dict[str, Any]]] = None) -> NeedSlotContract:
    active = detect_shape_template(prompt)
    domain_terms = extract_domain_terms(prompt)

    inverse_need = (
        "construct the missing operational slot implied by the prompt; "
        "select or generate only answers that preserve the required operation"
    )

    preserved = []
    if "TOOL" in active or "CONTRACT" in active:
        preserved.append("form contract before tool use")
    if "GROOVE" in active:
        preserved.append("shape model behavior through low-rank update without overwriting the base model")
    if "SEARCH" in active:
        preserved.append("retrieve by inverse operational fit when no noun match exists")
    if "REPAIR" in active:
        preserved.append("repair the failed dimension and rerun")
    if "MEMORY" in active:
        preserved.append("preserve trace continuity across turns rather than compressing state into summary text")
    if "RECURSE" in active:
        preserved.append("branch recursively until Ψ collapse or Ω residue")
    if not preserved:
        preserved.append("preserve the prompt's verb-level operation")

    boundaries = [
        "do not collapse on shared framework vocabulary alone",
        "require answer origin from the real model when model mode is enabled",
        "require prompt-grounded evidence for the selected answer",
        "prefer Ω over false Ψ when top branches disagree operationally",
        "preserve base answer when controller evidence is weak",
    ]

    return NeedSlotContract(
        prompt=prompt,
        active_templates=active,
        inverse_need=inverse_need,
        preserved_function="; ".join(preserved),
        boundary_conditions=boundaries,
        domain_carrier=domain_terms,
        forbidden_neighbors=infer_forbidden_neighbors(active),
        collapse_target="one executable answer with model origin, contract fit, prompt grounding, and trace sufficient to debug",
        repair_history=repair_history or [],
    )

def contract_to_text(c: NeedSlotContract) -> str:
    return (
        f"ACTIVE_TEMPLATES: {', '.join(c.active_templates)}\n"
        f"INVERSE_NEED: {c.inverse_need}\n"
        f"PRESERVED_FUNCTION: {c.preserved_function}\n"
        f"BOUNDARY_CONDITIONS: {' | '.join(c.boundary_conditions)}\n"
        f"DOMAIN_CARRIER: {', '.join(c.domain_carrier)}\n"
        f"FORBIDDEN_NEIGHBORS: {' | '.join(c.forbidden_neighbors)}\n"
        f"COLLAPSE_TARGET: {c.collapse_target}\n"
        f"REPAIR_HISTORY: {json.dumps(c.repair_history, ensure_ascii=False)}"
    )

def contract_field_terms(contract: NeedSlotContract) -> Dict[str, List[str]]:
    return {
        "need": words(contract.inverse_need, remove_nexus_surface=True),
        "function": words(contract.preserved_function, remove_nexus_surface=True),
        "boundary": words(" ".join(contract.boundary_conditions), remove_nexus_surface=True),
        "domain": contract.domain_carrier,
        "forbidden": words(" ".join(contract.forbidden_neighbors), remove_nexus_surface=True),
        "collapse": words(contract.collapse_target, remove_nexus_surface=True),
    }


## Branch generation

Construct ⊕ Verify ⊕ Repair ⊕ Counter.

v13 records origin:

- `model`
- `fallback`
- `fallback_error`

Only `model` can collapse to Ψ when `REQUIRE_MODEL_FOR_PSI=True`.


In [8]:
BRANCH_SYSTEMS = {
    "construct": (
        "You are the constructor branch. Build the answer from the missing operational slot first. "
        "Do not start with labels. Make the answer specific to the prompt."
    ),
    "verify": (
        "You are the verifier branch. Test the answer against need, function, boundary, trap, and collapse. "
        "Reject vocabulary agreement when operation differs."
    ),
    "repair": (
        "You are the repair branch. Identify the failed observable and patch only that dimension. "
        "Do not blanket-rewrite."
    ),
    "counter": (
        "You are the counter-branch. Name the strongest wrong path and explain why it fails. "
        "Then give the corrected path."
    ),
}

def deterministic_branch(prompt: str, contract: NeedSlotContract, branch_name: str, reason: str = "fallback") -> Dict[str, Any]:
    # Diagnostic fallback only. It cannot count as real Ψ in model-required mode.
    if branch_name == "construct":
        text = (
            f"Diagnostic fallback for construct. Prompt domain: {', '.join(contract.domain_carrier[:6])}. "
            f"Preserved function: {contract.preserved_function}. "
            "This is not a model answer; it exists only to keep the controller inspectable."
        )
    elif branch_name == "verify":
        text = (
            "Diagnostic fallback for verify. Check whether the candidate preserves the function, rejects forbidden neighbors, "
            "and stays grounded in the prompt rather than shared vocabulary."
        )
    elif branch_name == "repair":
        text = (
            "Diagnostic fallback for repair. If Ψ is blocked, extract Ω as the weakest observable and regenerate only that dimension."
        )
    elif branch_name == "counter":
        text = (
            "Diagnostic fallback for counter. The wrong path is collapsing on boilerplate or model failure while pretending success."
        )
    else:
        text = "Diagnostic fallback."

    return {
        "branch": branch_name,
        "answer": text,
        "origin": reason,
        "generation_error": MODEL_ERROR,
    }

def model_generate_one(prompt: str, contract: NeedSlotContract, branch_name: str) -> Dict[str, Any]:
    if not MODEL_GENERATION_READY:
        return deterministic_branch(prompt, contract, branch_name, reason="fallback_model_not_ready")

    system = BRANCH_SYSTEMS[branch_name]
    user = (
        "PROMPT:\n" + prompt.strip() + "\n\n"
        "NEED-SLOT CONTRACT:\n" + contract_to_text(contract) + "\n\n"
        "Rules:\n"
        "1. Do not say 'the prompt is asking'.\n"
        "2. Do not recite the pipeline unless the prompt asks for a pipeline.\n"
        "3. Use at least two prompt-domain terms, not just framework terms.\n"
        "4. Give one compact operational answer.\n"
        "5. Make the answer testable.\n"
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    try:
        text = raw_model_generate(messages, max_new_tokens=MAX_NEW_TOKENS)
        if not text.strip():
            return deterministic_branch(prompt, contract, branch_name, reason="fallback_empty_generation")
        return {
            "branch": branch_name,
            "answer": text.strip(),
            "origin": "model",
            "generation_error": None,
        }
    except Exception as e:
        err = "".join(traceback.format_exception_only(type(e), e)).strip()
        return {
            **deterministic_branch(prompt, contract, branch_name, reason="fallback_error"),
            "generation_error": err,
        }

def generate_candidates(prompt: str, contract: NeedSlotContract) -> List[Dict[str, Any]]:
    return [model_generate_one(prompt, contract, b) for b in BRANCH_SYSTEMS]


## Operational audit

Five observables:

$$
A_i = (F_{\text{need}}, F_{\text{function}}, F_{\text{boundary}}, F_{\text{trap}}, F_{\text{collapse}})
$$

v13 adds:

$$
F_{\text{prompt}},\quad P_{\text{boilerplate}},\quad O_{\text{model}}
$$


In [9]:
def answer_operational_audit(prompt: str, contract: NeedSlotContract, answer: str, origin: str) -> Dict[str, Any]:
    active = contract.active_templates
    fields = contract_field_terms(contract)
    a = str(answer).lower()

    F_need = clamp(0.55 * field_hit_score(answer, fields["need"]) + 0.45 * field_hit_score(answer, fields["domain"]))
    F_function = clamp(
        0.65 * field_hit_score(answer, fields["function"]) +
        0.35 * sum([
            contains_any(a, ["preserve", "maintain", "function", "operation", "before", "after"]),
            contains_any(a, ["execute", "candidate", "select", "verify", "state", "update"]),
        ]) / 2
    )
    F_boundary = clamp(
        0.60 * field_hit_score(answer, fields["boundary"]) +
        0.40 * sum([
            contains_any(a, ["boundary", "constraint", "gate", "reject", "protect", "forbidden"]),
            contains_any(a, ["false", "wrong", "weak", "premature", "surface", "boilerplate"]),
        ]) / 2
    )
    forbidden_hit = field_hit_score(answer, fields["forbidden"])
    trap_language = sum([
        contains_any(a, ["not", "instead", "wrong", "fails", "reject", "avoid", "forbidden"]),
        contains_any(a, ["tool-first", "surface", "generic", "threshold", "noun", "keyword", "boilerplate"]),
    ]) / 2
    F_trap = clamp(0.45 * forbidden_hit + 0.55 * trap_language)
    F_collapse = clamp(
        0.50 * field_hit_score(answer, fields["collapse"]) +
        0.50 * sum([
            contains_any(a, ["because", "therefore", "so", "result", "answer"]),
            contains_any(a, ["one", "single", "executable", "run", "test", "trace"]),
        ]) / 2
    )

    F_shape = shape_score(answer, active)
    F_prompt = field_hit_score(answer, words(prompt, remove_nexus_surface=True), max_terms=12)
    B_penalty = boilerplate_penalty(answer)
    O_model = 1.0 if origin == "model" else 0.0

    hot = clamp((F_need + F_function + F_shape + F_prompt) / 4)
    cold = clamp((F_boundary + F_trap + F_collapse + (1.0 - B_penalty)) / 4)
    hotcold_balance = clamp(1.0 - abs(hot - cold))

    quality_hmean = harmonic_mean([F_need, F_function, F_boundary, F_trap, F_collapse, max(0.05, F_prompt)])
    quality_mean = float(np.mean([F_need, F_function, F_boundary, F_trap, F_collapse, F_prompt]))

    return {
        "F_need": F_need,
        "F_function": F_function,
        "F_boundary": F_boundary,
        "F_trap": F_trap,
        "F_collapse": F_collapse,
        "F_shape": F_shape,
        "F_prompt": F_prompt,
        "boilerplate_penalty": B_penalty,
        "O_model": O_model,
        "hot": hot,
        "cold": cold,
        "hotcold_balance": hotcold_balance,
        "quality_hmean": quality_hmean,
        "quality_mean": quality_mean,
        "shape_mass": shape_mass(answer, active),
    }

def trace_sufficiency(answer: str, audit: Dict[str, Any], contract: NeedSlotContract) -> float:
    a = str(answer).lower()
    evidence_bits = [
        contains_any(a, ["because", "therefore", "so", "why", "means"]),
        contains_any(a, ["boundary", "constraint", "reject", "forbidden"]),
        contains_any(a, ["agent", "runtime", "model", "state", "candidate", "tool", "memory", "lora", "retrieval"]),
        contains_any(a, ["verify", "evidence", "trace", "audit", "test", "measure"]),
        audit["quality_hmean"] >= QUALITY_MIN,
        audit["F_prompt"] >= PROMPT_FIT_MIN,
    ]
    return clamp(sum(evidence_bits) / len(evidence_bits))

def contract_stance_score(contract: NeedSlotContract, answer: str) -> float:
    fields = contract_field_terms(contract)
    return float(np.mean([
        field_hit_score(answer, fields["need"]),
        field_hit_score(answer, fields["function"]),
        field_hit_score(answer, fields["boundary"]),
        field_hit_score(answer, fields["domain"]),
        field_hit_score(answer, fields["collapse"]),
    ]))

def branch_score(prompt: str, contract: NeedSlotContract, branch: Dict[str, Any]) -> Dict[str, Any]:
    audit = answer_operational_audit(prompt, contract, branch["answer"], branch["origin"])
    stance = contract_stance_score(contract, branch["answer"])
    trace = trace_sufficiency(branch["answer"], audit, contract)

    score = clamp(
        0.26 * audit["quality_hmean"] +
        0.18 * stance +
        0.16 * audit["F_shape"] +
        0.16 * trace +
        0.12 * audit["hotcold_balance"] +
        0.12 * audit["F_prompt"] -
        0.18 * audit["boilerplate_penalty"]
    )

    return {
        **branch,
        "score": score,
        "contract_stance": stance,
        "trace_sufficiency": trace,
        "audit": audit,
    }

def score_candidates(prompt: str, contract: NeedSlotContract, candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = [branch_score(prompt, contract, c) for c in candidates]
    flat = []
    for r in rows:
        a = r["audit"]
        flat.append({
            "branch": r["branch"],
            "origin": r["origin"],
            "score": r["score"],
            "contract_stance": r["contract_stance"],
            "trace_sufficiency": r["trace_sufficiency"],
            "quality_hmean": a["quality_hmean"],
            "F_need": a["F_need"],
            "F_function": a["F_function"],
            "F_boundary": a["F_boundary"],
            "F_trap": a["F_trap"],
            "F_collapse": a["F_collapse"],
            "F_shape": a["F_shape"],
            "F_prompt": a["F_prompt"],
            "boilerplate_penalty": a["boilerplate_penalty"],
            "O_model": a["O_model"],
            "hot": a["hot"],
            "cold": a["cold"],
            "hotcold_balance": a["hotcold_balance"],
            "generation_error": r.get("generation_error"),
            "answer": r["answer"],
            "detail": r,
        })
    df = pd.DataFrame(flat).sort_values("score", ascending=False).reset_index(drop=True)
    return df


## Gates and recursion

The controller now refuses fake Ψ.

Direct collapse requires:

$$
O_{\text{model}} = 1
$$

when model mode is enabled.


In [10]:
def audit_agreement(a: Dict[str, Any], b: Dict[str, Any]) -> float:
    keys = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse", "F_prompt"]
    diffs = [abs(float(a[k]) - float(b[k])) for k in keys]
    return clamp(1.0 - float(np.mean(diffs)))

def contract_stance_agreement(contract: NeedSlotContract, answer_a: str, answer_b: str) -> Dict[str, Any]:
    fields = contract_field_terms(contract)
    scores = {}
    for field_name in ["need", "function", "boundary", "domain", "collapse"]:
        terms = fields[field_name]
        sa = set(t for t in terms if str(t).lower() in str(answer_a).lower())
        sb = set(t for t in terms if str(t).lower() in str(answer_b).lower())
        if not terms:
            score = 0.5
        elif not sa and not sb:
            score = 0.10
        else:
            score = len(sa & sb) / max(1, len(sa | sb))
        scores[field_name] = clamp(score)
    aggregate = harmonic_mean(list(scores.values()))
    return {"aggregate": aggregate, "fields": scores}

def direct_collapse_gate(score_df: pd.DataFrame) -> Dict[str, Any]:
    top = score_df.iloc[0]
    margin = float(top["score"] - score_df.iloc[1]["score"]) if len(score_df) > 1 else float(top["score"])

    origin_ok = True
    if LOAD_REAL_MODEL and REQUIRE_MODEL_FOR_PSI:
        origin_ok = str(top["origin"]) == "model"

    ok = (
        float(top["score"]) >= PSI_MIN and
        margin >= MARGIN_MIN and
        float(top["trace_sufficiency"]) >= TRACE_MIN and
        float(top["quality_hmean"]) >= QUALITY_MIN and
        float(top["F_prompt"]) >= PROMPT_FIT_MIN and
        float(top["boilerplate_penalty"]) <= BOILERPLATE_MAX and
        origin_ok
    )

    failed = []
    if float(top["score"]) < PSI_MIN: failed.append("score")
    if margin < MARGIN_MIN: failed.append("margin")
    if float(top["trace_sufficiency"]) < TRACE_MIN: failed.append("trace")
    if float(top["quality_hmean"]) < QUALITY_MIN: failed.append("quality")
    if float(top["F_prompt"]) < PROMPT_FIT_MIN: failed.append("prompt_fit")
    if float(top["boilerplate_penalty"]) > BOILERPLATE_MAX: failed.append("boilerplate")
    if not origin_ok: failed.append("model_origin")

    return {
        "ok": bool(ok),
        "reason": "direct_model_grounded_collapse" if ok else "no_direct_collapse",
        "failed": failed,
        "margin": margin,
        "top_score": float(top["score"]),
        "top_origin": str(top["origin"]),
        "trace_sufficiency": float(top["trace_sufficiency"]),
        "quality_hmean": float(top["quality_hmean"]),
        "F_prompt": float(top["F_prompt"]),
        "boilerplate_penalty": float(top["boilerplate_penalty"]),
    }

def consensus_gate(contract: NeedSlotContract, score_df: pd.DataFrame) -> Dict[str, Any]:
    if len(score_df) < 2:
        return {"ok": False, "reason": "not_enough_candidates"}

    a = score_df.iloc[0]["detail"]
    b = score_df.iloc[1]["detail"]
    margin = float(score_df.iloc[0]["score"] - score_df.iloc[1]["score"])

    origin_ok = True
    if LOAD_REAL_MODEL and REQUIRE_MODEL_FOR_PSI:
        origin_ok = a["origin"] == "model" and b["origin"] == "model"

    lex_plain = jaccard_text(a["answer"], b["answer"], remove_nexus_surface=False)
    lex_operational = jaccard_text(a["answer"], b["answer"], remove_nexus_surface=True)
    audit_agree = audit_agreement(a["audit"], b["audit"])
    stance = contract_stance_agreement(contract, a["answer"], b["answer"])

    top_good = (
        score_df.iloc[0]["score"] >= PSI_MIN - 0.04 and
        score_df.iloc[1]["score"] >= PSI_MIN - 0.08 and
        score_df.iloc[0]["F_prompt"] >= PROMPT_FIT_MIN and
        score_df.iloc[1]["F_prompt"] >= PROMPT_FIT_MIN and
        score_df.iloc[0]["boilerplate_penalty"] <= BOILERPLATE_MAX and
        score_df.iloc[1]["boilerplate_penalty"] <= BOILERPLATE_MAX
    )

    ok = (
        origin_ok and
        top_good and
        margin <= 0.06 and
        audit_agree >= 0.82 and
        stance["aggregate"] >= 0.35 and
        lex_operational >= 0.10
    )

    return {
        "ok": bool(ok),
        "reason": "contract_anchored_model_consensus" if ok else "no_consensus",
        "origin_ok": bool(origin_ok),
        "margin": margin,
        "lex_plain": lex_plain,
        "lex_operational": lex_operational,
        "audit_agreement": audit_agree,
        "stance_agreement": stance["aggregate"],
        "stance_fields": stance["fields"],
    }

def extract_omega(score_df: pd.DataFrame, contract: NeedSlotContract, direct: Dict[str, Any], consensus: Dict[str, Any]) -> Dict[str, Any]:
    top = score_df.iloc[0]
    dims = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse", "F_shape", "F_prompt"]
    weakest_dim = min(dims, key=lambda k: float(top[k]))

    all_non_model = bool((score_df["origin"] != "model").all())
    reason = "model_generation_failed" if all_non_model and LOAD_REAL_MODEL else "failed_grounded_collapse"

    return {
        "reason": reason,
        "weakest_dim": weakest_dim,
        "weakest_value": float(top[weakest_dim]),
        "direct_failed": direct.get("failed", []),
        "top_branch": str(top["branch"]),
        "top_origin": str(top["origin"]),
        "top_score": float(top["score"]),
        "generation_errors": score_df[["branch", "origin", "generation_error"]].to_dict(orient="records"),
        "repair_instruction": f"repair {weakest_dim}; do not widen thresholds; regenerate with prompt-domain grounding",
    }

def repair_contract(contract: NeedSlotContract, omega: Dict[str, Any]) -> NeedSlotContract:
    hist = list(contract.repair_history)
    hist.append({
        "omega_reason": omega["reason"],
        "weakest_dim": omega["weakest_dim"],
        "instruction": omega["repair_instruction"],
    })
    return NeedSlotContract(
        prompt=contract.prompt,
        active_templates=contract.active_templates,
        inverse_need=contract.inverse_need,
        preserved_function=contract.preserved_function + f"; repair focus: {omega['weakest_dim']}",
        boundary_conditions=contract.boundary_conditions + [
            f"repair focus must address {omega['weakest_dim']}",
            "answer must name concrete prompt-domain terms",
            "answer must avoid boilerplate scaffold",
        ],
        domain_carrier=contract.domain_carrier,
        forbidden_neighbors=contract.forbidden_neighbors + ["boilerplate scaffold", "fallback answer", "fake model origin"],
        collapse_target=contract.collapse_target,
        repair_history=hist,
    )

def repair_candidates(prompt: str, contract: NeedSlotContract, omega: Dict[str, Any], previous_score_df: pd.DataFrame) -> List[Dict[str, Any]]:
    if omega["reason"] == "model_generation_failed":
        # Do not recurse into fake answers when model generation itself is broken.
        return previous_score_df["detail"].tolist()

    repair_prompt = (
        prompt.strip()
        + "\n\nREPAIR_FOCUS: " + omega["weakest_dim"]
        + "\nREPAIR_RULE: answer must be specific to this prompt and cannot use boilerplate."
    )
    return generate_candidates(repair_prompt, contract)

def krrb_recursive_resolve(
    prompt: str,
    contract: NeedSlotContract,
    candidates: List[Dict[str, Any]],
    depth: int = 0,
    trace: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:

    if trace is None:
        trace = []

    score_df = score_candidates(prompt, contract, candidates)
    direct = direct_collapse_gate(score_df)
    consensus = consensus_gate(contract, score_df)

    step = {
        "depth": depth,
        "contract": asdict(contract),
        "scores": score_df.drop(columns=["detail"]).to_dict(orient="records"),
        "direct_gate": direct,
        "consensus_gate": consensus,
    }
    trace.append(step)

    if direct["ok"]:
        return {
            "state": "Ψ",
            "reason": direct["reason"],
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    if consensus["ok"]:
        return {
            "state": "Ψ",
            "reason": consensus["reason"],
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    omega = extract_omega(score_df, contract, direct, consensus)
    step["omega"] = omega

    if omega["reason"] == "model_generation_failed":
        return {
            "state": "Ω",
            "reason": "model_generation_failed",
            "omega": omega,
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    if depth >= MAX_RECURSION_DEPTH:
        return {
            "state": "Ω",
            "reason": "max_recursion_depth",
            "omega": omega,
            "winner": score_df.iloc[0]["detail"],
            "score_df": score_df,
            "trace": trace,
            "depth": depth,
        }

    repaired_contract = repair_contract(contract, omega)
    repaired_candidates = repair_candidates(prompt, repaired_contract, omega, score_df)

    return krrb_recursive_resolve(
        prompt=prompt,
        contract=repaired_contract,
        candidates=repaired_candidates,
        depth=depth + 1,
        trace=trace,
    )


## Live runner

The key diagnostic is simple:

- If the model works, `origin=model` appears in the score table.
- If generation fails, state must be $\Omega$, not fake $\Psi$.


In [11]:
def save_json(obj: Dict[str, Any], path: Path):
    def default(x):
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, np.ndarray):
            return x.tolist()
        return str(x)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=default)

def run_rhi_v13(prompt: str, save: bool = True, show: bool = True) -> Dict[str, Any]:
    contract = build_contract(prompt)
    candidates = generate_candidates(prompt, contract)
    result = krrb_recursive_resolve(prompt, contract, candidates)

    final = {
        "run_id": RUN_ID,
        "prompt": prompt,
        "state": result["state"],
        "reason": result["reason"],
        "depth": result["depth"],
        "winner_branch": result["winner"]["branch"],
        "winner_origin": result["winner"]["origin"],
        "winner_score": float(result["score_df"].iloc[0]["score"]),
        "answer": result["winner"]["answer"],
        "contract": asdict(contract),
        "trace": result["trace"],
        "device_info": DEVICE_INFO,
        "model_ready": MODEL_READY,
        "model_generation_ready": MODEL_GENERATION_READY,
        "model_error": MODEL_ERROR,
        "model_id_or_path": MODEL_ID_OR_PATH,
    }

    if "omega" in result:
        final["omega"] = result["omega"]

    if save:
        prompt_id = hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:10]
        out_path = OUT_DIR / f"{RUN_ID}_{prompt_id}_result.json"
        rows_path = OUT_DIR / f"{RUN_ID}_{prompt_id}_score_rows.csv"
        save_json(final, out_path)
        result["score_df"].drop(columns=["detail"]).to_csv(rows_path, index=False)
        print("saved:", out_path)
        print("saved:", rows_path)

    if show:
        print("\nSTATE:", final["state"], "| REASON:", final["reason"], "| DEPTH:", final["depth"])
        print("WINNER:", final["winner_branch"], "| ORIGIN:", final["winner_origin"], "| SCORE:", round(final["winner_score"], 4))
        print("MODEL_READY:", MODEL_READY, "| MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
        if MODEL_ERROR:
            print("MODEL_ERROR:", MODEL_ERROR)
        print("\nCONTRACT\n--------")
        print(contract_to_text(contract))
        print("\nSCORES\n------")
        display(result["score_df"].drop(columns=["detail"]))
        if "omega" in final:
            print("\nOMEGA\n-----")
            print(json.dumps(final["omega"], indent=2, ensure_ascii=False))
        print("\nANSWER\n------")
        print(final["answer"])

    return final

LIVE_PROMPT = "explain why current AI agents fail when they use tools before forming a contract"
live_result = run_rhi_v13(LIVE_PROMPT, save=True, show=True)


saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_6e5b59d364_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_6e5b59d364_score_rows.csv

STATE: Ω | REASON: model_generation_failed | DEPTH: 0
WINNER: construct | ORIGIN: fallback_model_not_ready | SCORE: 0.4828
MODEL_READY: True | MODEL_GENERATION_READY: False
MODEL_ERROR: AttributeError

CONTRACT
--------
ACTIVE_TEMPLATES: CONTRACT, TOOL
INVERSE_NEED: construct the missing operational slot implied by the prompt; select or generate only answers that preserve the required operation
PRESERVED_FUNCTION: form contract before tool use
BOUNDARY_CONDITIONS: do not collapse on shared framework vocabulary alone | require answer origin from the real model when model mode is enabled | require prompt-grounded evidence for the selected answer | prefer Ω over false Ψ when top branches disagree operationally | preserve base answer when controller evidence is weak
DOMAIN_CARRIER: explain, curr

,branch,origin,score,contract_stance,trace_sufficiency,quality_hmean,F_need,F_function,F_boundary,F_trap,F_collapse,F_shape,F_prompt,boilerplate_penalty,O_model,hot,cold,hotcold_balance,generation_error,answer
0,construct,fallback_model_not_ready,0.482821,0.504444,0.333333,3.734765e-01,0.472222,0.825,0.18,0.320,0.361111,0.266667,0.777778,0.0,0.0,0.585417,0.465278,0.879861,AttributeError,Diagnostic fallback for construct. Prompt doma...
1,verify,fallback_model_not_ready,0.194683,0.062222,0.500000,6.000000e-09,0.061111,0.350,0.32,0.275,0.000000,0.100000,0.000000,0.0,0.0,0.127778,0.398750,0.729028,AttributeError,Diagnostic fallback for verify. Check whether ...
2,counter,fallback_model_not_ready,0.144433,0.106667,0.166667,6.000000e-09,0.100000,0.000,0.32,0.550,0.055556,0.000000,0.222222,0.0,0.0,0.080556,0.481389,0.599167,AttributeError,Diagnostic fallback for counter. The wrong pat...
3,repair,fallback_model_not_ready,0.117833,0.066667,0.000000,2.000000e-09,0.172222,0.000,0.20,0.000,0.000000,0.000000,0.111111,0.0,0.0,0.070833,0.300000,0.770833,AttributeError,Diagnostic fallback for repair. If Ψ is blocke...



OMEGA
-----
{
  "reason": "model_generation_failed",
  "weakest_dim": "F_boundary",
  "weakest_value": 0.18,
  "direct_failed": [
    "score",
    "trace",
    "quality",
    "model_origin"
  ],
  "top_branch": "construct",
  "top_origin": "fallback_model_not_ready",
  "top_score": 0.4828205502998704,
  "generation_errors": [
    {
      "branch": "construct",
      "origin": "fallback_model_not_ready",
      "generation_error": "AttributeError"
    },
    {
      "branch": "verify",
      "origin": "fallback_model_not_ready",
      "generation_error": "AttributeError"
    },
    {
      "branch": "counter",
      "origin": "fallback_model_not_ready",
      "generation_error": "AttributeError"
    },
    {
      "branch": "repair",
      "origin": "fallback_model_not_ready",
      "generation_error": "AttributeError"
    }
  ],
  "repair_instruction": "repair F_boundary; do not widen thresholds; regenerate with prompt-domain grounding"
}

ANSWER
------
Diagnostic fallback for construc

In [12]:
# Batch tests. These are the same v11 prompts so the comparison is clean.

TEST_PROMPTS = [
    "explain why current AI agents fail when they use tools before forming a contract",
    "how should a LoRA adapter train a slot-builder without overwriting the base model",
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
    "fix a recursive AI controller that collapses because two branches share vocabulary but disagree operationally",
    "explain memory in an agent as trace continuity rather than a text summary",
]

batch = []
for p in TEST_PROMPTS:
    print("\n" + "="*100)
    print("PROMPT:", p)
    try:
        r = run_rhi_v13(p, save=True, show=False)
        batch.append({
            "prompt": p,
            "state": r["state"],
            "reason": r["reason"],
            "depth": r["depth"],
            "winner_branch": r["winner_branch"],
            "winner_origin": r["winner_origin"],
            "winner_score": r["winner_score"],
            "model_generation_ready": r["model_generation_ready"],
            "answer_preview": r["answer"][:120].replace("\n", " "),
        })
    except Exception as e:
        batch.append({
            "prompt": p,
            "state": "ERROR",
            "reason": "".join(traceback.format_exception_only(type(e), e)).strip(),
            "depth": None,
            "winner_branch": None,
            "winner_origin": None,
            "winner_score": None,
            "model_generation_ready": MODEL_GENERATION_READY,
            "answer_preview": "",
        })
        print(traceback.format_exc())

batch_df = pd.DataFrame(batch)
display(batch_df)

summary_path = OUT_DIR / f"{RUN_ID}_batch_summary.csv"
batch_df.to_csv(summary_path, index=False)
print("saved:", summary_path)



PROMPT: explain why current AI agents fail when they use tools before forming a contract
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_6e5b59d364_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_6e5b59d364_score_rows.csv

PROMPT: how should a LoRA adapter train a slot-builder without overwriting the base model
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_84a5615f0a_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_84a5615f0a_score_rows.csv

PROMPT: design a shape-first retrieval step where no noun match exists but the inverse need is clear
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_a8ce36a582_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_a8ce36a582_score_rows.csv

PROMPT: fix a recursive AI controller that collapses because two branches share vocabulary but disagree operationally
saved: 

,prompt,state,reason,depth,winner_branch,winner_origin,winner_score,model_generation_ready,answer_preview
0,explain why current AI agents fail when they u...,Ω,model_generation_failed,0,construct,fallback_model_not_ready,0.482821,False,Diagnostic fallback for construct. Prompt doma...
1,how should a LoRA adapter train a slot-builder...,Ω,model_generation_failed,0,construct,fallback_model_not_ready,0.510592,False,Diagnostic fallback for construct. Prompt doma...
2,design a shape-first retrieval step where no n...,Ω,model_generation_failed,0,construct,fallback_model_not_ready,0.546277,False,Diagnostic fallback for construct. Prompt doma...
3,fix a recursive AI controller that collapses b...,Ω,model_generation_failed,0,construct,fallback_model_not_ready,0.487327,False,Diagnostic fallback for construct. Prompt doma...
4,explain memory in an agent as trace continuity...,Ω,model_generation_failed,0,construct,fallback_model_not_ready,0.602938,False,Diagnostic fallback for construct. Prompt doma...


saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v13_outputs\rhi_v13_29cb18de15_batch_summary.csv


## What to look for

Δ **Good signs**

- `MODEL_GENERATION_READY=True`
- `origin=model`
- `state=Ψ`
- `reason=direct_model_grounded_collapse` or `contract_anchored_model_consensus`
- `depth` may be `0` for easy prompts, but not all prompts should collapse by boilerplate.
- Winner answer is prompt-specific.

Ω **Bad signs**

- `MODEL_GENERATION_READY=False`
- `state=Ω`, `reason=model_generation_failed`
- `origin=fallback_error`
- Repeated generic answer scaffolds.
- `F_prompt` low while `F_shape` high.

↻ **Next fold**

If v13 produces real model-origin Ψ, then we can train the slot-builder on:

$$
Q \rightarrow C_Q
$$

not on direct answering. That is the correct groove target.
